# Module 05 — Notebook 3: Heatmaps and Subplots

## Learning Objectives

By the end of this notebook you will be able to:
- Create a heatmap to visualize a model × task scorecard
- Build multi-panel figures with `plt.subplots(nrows, ncols)`
- Share axes across subplots for clean comparisons
- Add a color bar and control color range with `vmin` / `vmax`
- Compose a complete analysis figure from multiple chart types

**Time:** ~20 minutes

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_length
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

%matplotlib inline

DATA_PATH = Path("../../data/synthetic/evaluation_results.csv")
df = pd.read_csv(DATA_PATH)
print("Loaded:", df.shape)

## 1. Heatmaps — Visualizing a Matrix

A heatmap encodes numeric values as colors in a grid. It's the natural visualization for
a **model × task scorecard** — the reader can instantly spot where each model is weak.

seaborn's `sns.heatmap()` takes a 2D DataFrame (like a pivot table) and colors each cell:

```python
pivot = df.pivot_table(index="model", columns="task", values="score")
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.5, vmax=1.0, ax=ax)
```

Key parameters:
- `annot=True` — show the numeric value in each cell
- `fmt=".2f"` — format the annotation as a 2-decimal float
- `cmap` — color map; `"RdYlGn"` (red-yellow-green) is intuitive for scores
- `vmin`, `vmax` — fix the color scale range (important for consistency)

In [ ]:
pivot = df.pivot_table(index="model", columns="task", values="score")
print("Pivot shape:", pivot.shape)

fig, ax = plt.subplots(figsize=(10, 4))

sns.heatmap(
    pivot,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0.5, vmax=1.0,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Model × Task Evaluation Scorecard", fontsize=13, pad=12)
ax.set_xlabel("Task")
ax.set_ylabel("Model")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

## 2. Multi-Panel Figures

Pass `nrows` and `ncols` to `plt.subplots()` to get a grid of axes:

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))   # 1 row, 2 columns
fig, axes = plt.subplots(2, 2, figsize=(10, 8))   # 2×2 grid
```

`axes` becomes a NumPy array of `Axes` objects. Index them with `axes[0]`, `axes[1]`, or
for 2D grids, `axes[row, col]`.

> **`sharey=True`**: forces all subplots to share the same y-axis range — essential
> when you want the reader to compare magnitudes across panels.

In [ ]:
models = df["model"].unique()
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)

task_order = sorted(df["task"].unique())

for ax, model in zip(axes, models):
    model_df = df[df["model"] == model].set_index("task")["score"].reindex(task_order)
    colors = ["#27AE60" if s >= 0.85 else "#E74C3C" if s < 0.7 else "#F39C12"
              for s in model_df.values]
    ax.barh(task_order, model_df.values, color=colors)
    ax.axvline(0.8, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_title(model, fontsize=9)
    ax.set_xlim(0.5, 1.05)
    if ax != axes[0]:
        ax.set_yticklabels([])

axes[0].set_ylabel("Task")
fig.suptitle("Per-Model Task Scores (green ≥ 0.85, orange = mid, red < 0.70)",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 3. Annotating a Heatmap — Highlighting Outliers

You can add custom text annotations to specific cells using `ax.text()` after drawing the heatmap.
Here we add a warning marker to any cell below 0.70.

In [ ]:
pivot = df.pivot_table(index="model", columns="task", values="score")

fig, ax = plt.subplots(figsize=(11, 4))

sns.heatmap(
    pivot, annot=True, fmt=".2f",
    cmap="RdYlGn", vmin=0.5, vmax=1.0,
    linewidths=0.5, ax=ax
)

# Add a star to cells below 0.70 — critical failures
for row_i, model in enumerate(pivot.index):
    for col_i, task in enumerate(pivot.columns):
        val = pivot.loc[model, task]
        if val < 0.70:
            ax.text(col_i + 0.5, row_i + 0.15, "⚠",
                    ha="center", va="center", color="black", fontsize=14)

ax.set_title("Scorecard — ⚠ marks scores below 0.70 (critical failures)", fontsize=12)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

---
## Your Turn — Exercise 1: Difference Heatmap

Build a **difference heatmap** showing how much each v2 model improved over v1.

1. Create `diff_pivot` — a DataFrame with two rows (`model-a`, `model-b`) and 5 columns (tasks).
   Each cell = v2 score − v1 score for that family/task.
   > **Hint:** Build pivots for v1 and v2, then subtract.
2. Store the improvement for model-a on `"honesty_calibration"` in `a_honesty_gain`,
   rounded to 2 decimal places.
3. Create a heatmap of `diff_pivot` using `cmap="RdBu"` (diverging: red=worse, blue=better)
   with `center=0` so zero is white.

In [ ]:
# YOUR CODE HERE
diff_pivot     = None   # DataFrame: rows = model families, cols = tasks, values = v2 - v1
a_honesty_gain = None   # model-a improvement on honesty_calibration, rounded to 2 dp

fig, ax = None, None
# sns.heatmap(diff_pivot, ..., cmap="RdBu", center=0, annot=True, fmt=".2f")
# plt.tight_layout(); plt.show()

In [ ]:
check_type(diff_pivot, pd.DataFrame, "diff_pivot is a DataFrame")
check_equal(diff_pivot.shape, (2, 5), "diff_pivot has 2 rows and 5 columns")
# model-a honesty: v2=0.91 minus v1=0.88 = 0.03
check_approx(a_honesty_gain, 0.03, 1e-2, "a_honesty_gain")

---
## Your Turn — Exercise 2: 2×2 Summary Figure

Build a 2×2 figure combining four chart types.

1. Create `fig, axes = plt.subplots(2, 2, figsize=(12, 9))`.
2. `axes[0, 0]`: bar chart of per-model mean scores.
3. `axes[0, 1]`: horizontal bar chart of per-task mean scores.
4. `axes[1, 0]`: seaborn boxplot of scores by model.
5. `axes[1, 1]`: heatmap of the model × task scorecard.
6. Add `fig.suptitle("Evaluation Summary", fontsize=14)`.
7. Store the number of axes in `n_axes` (should be 4).

> **Hint:** `axes` from a 2×2 subplots call has shape `(2, 2)` — access panels as `axes[0, 0]`.

In [ ]:
pivot = df.pivot_table(index="model", columns="task", values="score")
per_model = df.groupby("model")["score"].mean().sort_values(ascending=False)
per_task  = df.groupby("task")["score"].mean().sort_values()

# YOUR CODE HERE
n_axes = None   # integer: total number of Axes in the figure

fig, axes = None, None
# Build the 4-panel figure...
# plt.tight_layout(); plt.show()

In [ ]:
check_equal(isinstance(fig, plt.Figure), True, "fig is a Figure")
check_equal(int(n_axes), 4, "figure has 4 axes")

---
## Why This Matters for AI Research Engineering

The heatmap is the **most important plot** in model evaluation. Every benchmark paper, internal scorecard, and model comparison report uses a variant of the model × task matrix. Being able to generate it from a DataFrame in 5 lines of code — and then customize it to highlight failures — is a core skill.

The difference heatmap (v2 − v1) is how you report **what changed** in a new model version. A cell that's deep red means a regression — something your safety team needs to investigate.

Multi-panel figures (2×2 or 1×4) let you pack a complete analysis into a single image for a research memo or PR review. Rather than attaching four separate plots, one tight figure tells the whole story.

## Summary

| What | Code |
|------|------|
| Heatmap | `sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn")` |
| Color range | `vmin=0.5, vmax=1.0` |
| Diverging colors | `cmap="RdBu", center=0` |
| Multi-panel | `fig, axes = plt.subplots(nrows, ncols, figsize=(w, h))` |
| Shared y-axis | `sharey=True` |
| Access panel | `axes[0]` (1D) or `axes[row, col]` (2D) |
| Figure title | `fig.suptitle("...", fontsize=14)` |

**Next:** Notebook 4 — the eval plots mini-project: save a complete analysis figure set to `output/plots/`.